In [1]:
### Quick intros to python topics

In [1]:
import math
math.atan2(10, 10) * 180/math.pi

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
# print(QLineF().angle())

if QLineF().isNull(): 
    print('null')
if QLineF(0,0,0,0).isNull(): 
    print('null')
if QLineF(10,10,10,10).isNull(): 
    print('null')
    

from utils import Utils 
p1, p2, p3 = QPoint(0,0) , QPoint(10,0) , QPoint(0,10) #  This should be cw aka 1
ori = Utils.threePointOrientation(p1,p2,p3)
print('Ori:', ori) # ccw all checks out , but the QT Coordinate system is flipped in y 

p4,p5,p6 = QPoint(0,0) , QPoint(10,0) , QPoint(0 , -10) # This should be ccw aka 2 
ori = Utils.threePointOrientation(p4,p5,p6)
print('Ori2:', ori)




null
null
null
Ori: 1
Ori2: 2


In [3]:
https://stackoverflow.com/questions/13102787/prevent-qgraphicsitem-from-moving-outside-of-qgraphicsscene
This SO comment makes a case for why youd use .itemChange rather than mouseMOveEvent: Ex in multiply selected items, only one item receives a mouseMoveEvent

SyntaxError: invalid syntax (2398614847.py, line 1)

In [ ]:
# I think I am using angles bad, try using atan2

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
from LayersItem import *
from MyView import MyView
from BoardScene import BoardScene
from utils import Utils
from collections import defaultdict
import sys
from utils import CopperItemContainer ,LayerItem
from MainWindow import MainWindow

class ViaBase():#QGraphicsItem):
    def __init__(self, outerDiameter, innerDiameter, clearance, **kwargs):#,parent=None):
        # print('VIABASE.KWARGS:', kwargs)
        super().__init__(**kwargs)#parent) 
        # super().__init__(parent=parent) TypeError: NO BAD no keywords use positional: LayersContainer.__init__() got an unexpected keyword argument 'parent' # IDK why this happens-- could not replicate in simple example. Something about QGraphicsItem preferring positional args. But sometimes it can take kwargs. I always put classes, which inherit QGI, LAST, in the inheritance, because super() cannot propagate correctly after it hits QGI.
        
        self._outerDiameter = outerDiameter 
        self._innerDiameter = innerDiameter 
        self._clearance = clearance

        self._boundingRect = QRectF(-(outerDiameter+clearance)/2 , -(outerDiameter+clearance)/2 , outerDiameter+clearance , outerDiameter+clearance) # Must include clearance in BR so we can redraw the clearance w/o artifacts.

    def boundingRect(self):
        return self._boundingRect 

    # def paint(self, painter, option, widget):
    #     pass 
    
    def shape(self): # Note that shape, .bR, are GOING to be reimplementing QGI.shape,.bR once ViaBase is inherited by Via, ViaItem. (Good, bad) practice? 
        path = QPainterPath()
        path.addEllipse(QPoint(),  self.outerRadius(), self.outerRadius() ) # This doesnt account for clearance, and it doesnt need to, but bR needs to
        return path 
        
    def outerDiameter(self):
        return self._outerDiameter 
    def innerDiameter(self):
        return self._innerDiameter 

    def outerRadius(self):
        return self._outerDiameter/2
    def innerRadius(self):
        return self._innerDiameter/2
    
    def clearance(self):
        return self._clearance
            
# class ViaItem(CopperItem, ViaBase):
class ViaItem(LayerItem, ViaBase, QGraphicsItem):
        # def QGI.__init__(self, parent: PySide6.QtWidgets.QGraphicsItem | None= ...) -> None: ...
    def __init__(self, layer, outerDiameter, innerDiameter, clearance, color , parent):
        # super().__init__(outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, parent=parent)
        super().__init__( layer, outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, parent=parent) # TypeError: ViaBase.__init__() takes 4 positional arguments but 5 were given
        # QGraphicsItem.__init__(self, parent)
        # print('VIAITEM.LAYER:', self.layer())
        self._color = color
        # self._boundingRect = QRectF(-(outerDiameter+clearance)/2 , -(outerDiameter+clearance)/2 , outerDiameter+clearance , outerDiameter+clearance) # Must include clearance in BR so we can redraw the clearance w/o artifacts.

    # def boundingRect(self): # Belive this covered by super
    #     return self._boundingRect
    
    def paint(self, painter, option, widget): # QGraphicsItem.paint reimplementation to draw the aspects of a via: a clearance indicator and some colored circles
        #DrawClearance
        painter.setPen(QPen(self._color, 0))
        painter.setBrush(Qt.BrushStyle.NoBrush)
        painter.drawEllipse(QPointF(), self.clearance()/2, self.clearance()/2)
        #DrawVia
        painter.setPen(Qt.NoPen)
        painter.setBrush(QBrush(self._color, bs=Qt.BrushStyle.SolidPattern))
        painter.drawEllipse(QPoint(0,0), self.outerDiameter()/2 , self.outerDiameter()/2)
        painter.setBrush(QColor(230,230,230)) # Gray
        painter.drawEllipse(QPoint(0,0), self.innerDiameter()/2+Utils.viaPlatingThickness , self.innerDiameter()/2+Utils.viaPlatingThickness)
        painter.setBrush(QColor(255,215,0)) # Gold
        painter.drawEllipse(QPoint(0,0), self.innerDiameter()/2, self.innerDiameter()/2)

    def connectedNets(self): 
        hitIds = self.scene().rtrees()[self.layer()].intersection(self.sceneBounds)
# class Via(LayersContainer, ViaBase): # A Via is made up of several childItem viaItems, one viaItem per layer. 
class Via(ViaBase, CopperItemContainer, QGraphicsItem):
    
    def __init__(self,  outerDiameter, innerDiameter, clearance=Utils.viaClearance, layers=Utils.CopperLayers): # layers : A via may exist on all or some layers, default all # clearance: default 1mm
        # print('VIA.MRO:', Via.mro())
        super().__init__(outerDiameter=outerDiameter, innerDiameter=innerDiameter, clearance=clearance, layers = layers)

        self.setFlags(QGraphicsItem.ItemIsMovable | QGraphicsItem.ItemIsSelectable)
        
        # print('VIA.LAYERS():', self.layers())
        for layer in self.layers():
            # print('LAYER:', layer)
            ViaItem(layer, outerDiameter, innerDiameter, clearance, Utils.layerColors[layer], self)
            # self.copperItems()[layer].append(item) Phasing out# Track ViaItem as copperItems ( is this necessary? )

    def nearestSceneSnap(self, pos): # Via only has one snap; center. But since TraceItem has two snaps, all items need this method to maintain consistent API.
        return self.scenePos()
    
    def mouseMoveEvent(self, event):
        super().mouseMoveEvent(event)
                
        self.setSceneTerminal()
        
    def net(self):
        return self._net 
    def setNet(self, net):
        # Note pad net is determined by the schematic connections. Via,Trace, net is determined by pads
        self._net = net
    def sceneTerminal(self): 
        return self.scenePos()
    def setSceneTerminal(self):
        self._sceneTerminal = self.scenePos()
        
    def sceneTerminals(self):
        return self._sceneTerminals
    def setSceneTerminals(self):
        self.setSceneTerminal()
        self._sceneTerminals = [self.sceneTerminal()]
    def boundingRect(self):
        return self.childrenBoundingRect() or QRectF()
    def paint(self, painter, option, widget):
        pass# ChildrenItems will paint themselves

from NetSymbol import NetSymbol
from Trace import Trace

window = MainWindow()
boardScene = window.centralWidget().widget(1).scene()
via = Via(50, 30, layers = ['F.Cu', 'B.Cu'])
trace = Trace(50,50 , 1000, 800, layers = ['F.Cu'], traceWidth = 1)
trace.setLine(QLineF(0,0, 100,100))
# print('TRACE.LINE():', trace.line())
boardScene.addItem(trace)

# print([item for item in via.childItems()])
ns = NetSymbol('?', 1, "symbols/NetSymbols/GND.sym")

via.setPos(100,100)
boardScene.addItem(via)
window.show()
sys.exit(qApp.exec())
# SELF.COMPONENTS: defaultdict(<class 'collections.defaultdict'>, {'?': defaultdict(None, {1: <Component.Component object at 0x00000272C7C6D790>}), 'GND': defaultdict(None, {})})


In [1]:
print(QLineF().angle())

NameError: name 'QLineF' is not defined

In [1]:
270.00 + 135 -360

45.0

(-3, -6)

In [ ]:
print (5//1)
print(5// 2)

print(5%2)
print(5.3%3)

print(45.000004 % 45)

In [ ]:
Trace adjust is not precise. Needs new strategy. (slope function had float inaccuracy, plus switched from 1e3 to 1e9 ) Ex trace laid at multiple of 45 degrees. but after dragging , so slightly off of 45n degrees. Really bad. Also it don't work for perp connections.
Traces must avoid different nets 



In [ ]:
print(0 is 0)

In [ ]:
# TraceBase, TraceItem, and Trace, from QGraphicsItem

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
import sys
from utils import app, Utils, CopperItemContainer
from BoardScene import BoardScene
from BoardView import BoardView
from MainWindow import MainWindow

class TraceBase(): 
    def __init__(self, x1, y1, x2, y2, traceWidth, *args, **kwargs ): 
        super().__init__(*args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 
        self._traceWidth = traceWidth

        # self.setPen(QPen(Qt.black, traceWidth , c = Qt.PenCapStyle.RoundCap))
        # self.setBrush(QBrush(Qt.black))

    def boundingRect(self): 
        width = self.x2()- self.x1()
        height= self.y2() - self.y1() 
        
        return QRectF( *self.p1() , width, height).normalized().adjusted(-self._traceWidth/2 , -self._traceWidth/2 , self._traceWidth/2 , self._traceWidth/2)

        
    def shape(self):
        path = QPainterPath()
        path.moveTo(QPointF(*self.p1()))
        path.lineTo(QPointF(*self.p2()))
        stroker = QPainterPathStroker() # In computer graphics, 'stroking' is the known difficult problem of offsetting shapes. Qt uses it to calculate 'fillable outlines of shapes': Give a path, get an offset of that path. note it strokes the 'inside' and 'outside' of the given shape, so there's two
        stroker.setWidth(self.traceWidth()) 
        stroker.setJoinStyle(Qt.RoundJoin)
        stroker.setCapStyle(Qt.RoundCap)        
        path = stroker.createStroke(path)
        return path 
    
    def traceWidth(self): 
        return self._traceWidth 
        
    def pen(self):
        return self._pen
    
    # def brush(self): QGLI has no brush. Trace should have no brush as well 
    #     return self._brush 

    def x1(self) :
        return self._x1 
    def y1(self): 
        return self._y1 
    def x2(self): 
        return self._x2 
    def y2(self): 
        return self._y2
    
    def p1(self):
        return (self._x1, self._y1)
    def setP1(self, p1 ): 
        self.prepareGeometryChange()
        self._x1 , self._y1 = p1 
        
    def p2(self):
        return (self._x2 , self._y2)
    def setP2(self, p2): 
        self.prepareGeometryChange()
        self._x2 , self._y2 = p2

    def line(self):
        return (self.p1() , self.p2() )    
    def setLine(self, x1=None, y1=None, x2=None , y2=None , p1=None , p2=None , line=None ): 
        self.prepareGeometryChange()
        
        if x1 and y1 and x2 and y2: 
            self._x1 = x1 
            self._y1 = y1 
            self._x2 = x2
            self._y2 = y2
            
        elif p1 and p2: 
            self._x1 = p1[0]
            self._y1 = p1[1] 
            self._x2 = p2[0] 
            self._y2 = p2[1] 
            
        elif line: 
            if len(line)==4: 
                self._x1 = line[0]
                self._y1 = line[1]
                self._x2 = line[2] 
                self._y2 = line[3]

class TraceItem(TraceBase, QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, layer, traceWidth, parent, *args, **kwargs ): 
        super().__init__(x1, y1, x2, y2, traceWidth, parent=parent, *args, **kwargs)

        self._layer = layer

        self.setPen(QPen(Utils.layerColors[layer], traceWidth , c = Qt.PenCapStyle.RoundCap))

        # self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)
            
    def paint(self, painter, option, widget): 
        painter.setPen(self.pen())
        painter.drawLine(self.x1() , self.y1() , self.x2() , self.y2())
    
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
    def pen(self):
        return self._pen
    def setPen(self, pen): 
        self.prepareGeometryChange() 
        self._traceWidth = pen.width() 
        self._pen = QPen(Utils.layerColors[self._layer] , pen.width(), c = Qt.PenCapStyle.RoundCap)



class Trace(TraceBase, CopperItemContainer, QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, layers, traceWidth, parent=None, *args, **kwargs ): 
        super().__init__(x1=x1, y1=y1, x2=x2, y2=y2, layers = layers, traceWidth=traceWidth, parent=parent, *args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 
        self._traceWidth = traceWidth

        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)

        for layer in self.layers(): 
            traceItem = TraceItem(self._x1 , self._y1 , self._x2 ,self._y2 , layer, traceWidth, self) 
            
    def paint(self, painter, option ,widget): 
        pass 

    def setLine(self, line): 
        super().setLine(line) # TraceBase.setLine
        for child in self.childItems(): #Control child TraceItem lines
            if isinstance(child, TraceItem): 
                child.setLine(line) 
                
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
        for child in self.childItems(): # Control child TraceItems 
            if isinstance(child, TraceItem): 
                child.setTraceWidth(traceWidth)

    def sceneTerminals(self):
        return self._sceneTerminals
    def setSceneTerminals(self):
        self._sceneTerminals = [self.p1() , self.p2()]





scene = BoardScene()
view = BoardView() 

originMarker = QGraphicsEllipseItem(-10,-10,20,20) 
originMarker.setPen(QPen(Qt.magenta , 1))
scene.addItem(originMarker)

# trace = TraceItem(30,50, 100,100 ,'F.Cu')
traceWidth = 1 
trace = Trace(30,50 ,100,100, ['B.Cu', 'F.Cu'] , traceWidth)
# trace.setLine(100,100, 200,200)
scene.addItem(trace)

view.setScene(scene) 
view.show() 
sys.exit(app.exec())

        

In [ ]:
# Here's why we cannot(reasonably) subclass QGraphicsLineItem for use in Trace: It cannot handle **kwargs input 
# This is also why TraceBase does not inherit QGraphicsItem; why Trace&TraceItem each inherit QGraphicsItem directly
# unfortunately you DO have to master cooperative inheritance/multiple inheritance to really understand this :(
from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *

# class Trace(QGraphicsLineItem):
#     def __init__(self, *args, **kwargs): 
#         super().__init__(*args, **kwargs)

# class A(Trace): 
#     def __init__(self, a, *args, **kwargs):
#         print('A.ARGS:', args)
#         print('A.KWARGS:', kwargs)
#         super().__init__(*args, **kwargs) 

# a = A('a',  line= QLineF())
# print(a.line())



# However, we need not forward kwargs to the constructor. We can super().__init__() without any arguements, then use setters post-constructor.
class Trace(QGraphicsLineItem):
    def __init__(self, **kwargs):
        super().__init__()  # Call parent with no args
        self.setLine(kwargs['line']) # Use setter method post-constructor. We need not forward kwargs to the constructor, which cannot handle kwarg forwarding


class A(Trace):
    def __init__(self, a, **kwargs):
        print('A.ARGS:', a)
        print('A.KWARGS:', kwargs)
        super().__init__(**kwargs)

# problem: argument is 'hiding' in kwargs. Terrible practice? Workaround : don't use QGLI, diy QGI. This is hard, however. need prepareGeometryChange, lots of getters/setters. 

a = A('a', line=QLineF(10,10,20,20))
print(a.line())

In [ ]:
# Create TraceItem from QGraphicsItem. All this work just to use *args/**kwargs? 

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
import sys
from utils import app

class TraceItem(QGraphicsItem): 
    def __init__(self, x1, y1, x2, y2, parent=None, *args, **kwargs ): 
        super().__init__(*args, **kwargs)
        self._x1 = x1 
        self._y1 = y1
        self._x2 = x2
        self._y2 = y2 


        self.setPen(QPen(Qt.black, 1 , c = Qt.PenCapStyle.RoundCap))
        self.setBrush(QBrush(Qt.black))

        self.setFlags(QGraphicsItem.GraphicsItemFlag.ItemIsSelectable | QGraphicsItem.ItemIsMovable)
        

    def boundingRect(self): 
        width = self.x2()- self.x1()
        height= self.y2() - self.y1() 
        
        return QRectF( *self.p1() , width, height).normalized().adjusted(-self._traceWidth/2 , -self._traceWidth/2 , self._traceWidth/2 , self._traceWidth/2)
    
    def paint(self, painter, option, widget): 
        painter.setPen(self.pen())
        painter.setBrush(self.brush())
        painter.drawLine(self.x1() , self.y1() , self.x2() , self.y2())
        
    def shape(self):
        path = QPainterPath()
        path.moveTo(QPointF(*self.p1()))
        path.lineTo(QPointF(*self.p2()))
        stroker = QPainterPathStroker() # In computer graphics, 'stroking' is the known difficult problem of offsetting shapes. Qt uses it to calculate 'fillable outlines of shapes': Give a path, get an offset of that path. note it strokes the 'inside' and 'outside' of the given shape, so there's two
        stroker.setWidth(self.traceWidth()) 
        stroker.setJoinStyle(Qt.RoundJoin)
        stroker.setCapStyle(Qt.RoundCap)        
        path = stroker.createStroke(path)
        return path 
    
    def traceWidth(self): 
        return self._traceWidth 
    def setTraceWidth(self, traceWidth): 
        self.prepareGeometryChange() 
        self._traceWidth = traceWidth
        self._pen = QPen(self.pen().color() , traceWidth, c = Qt.PenCapStyle.RoundCap)
        
    def pen(self):
        return self._pen
    def setPen(self, pen): 
        self.prepareGeometryChange() 
        self._traceWidth = pen.width() 
        self._pen = QPen(pen.color() , pen.width(), c = Qt.PenCapStyle.RoundCap)

    def brush(self): 
        return self._brush 
    def setBrush(self, brush): 
        self._brush = brush

    def x1(self) :
        return self._x1 
    def y1(self): 
        return self._y1 
    def x2(self): 
        return self._x2 
    def y2(self): 
        return self._y2
    
    def p1(self):
        return (self._x1, self._y1)
    def setP1(self, p1 ): 
        self.prepareGeometryChange()
        self._x1 , self._y1 = p1 
        
    def p2(self):
        return (self._x2 , self._y2)
    def setP2(self, p2): 
        self.prepareGeometryChange()
        self._x2 , self._y2 = p2

    def line(self):
        return (self.p1() , self.p2() )    
    def setLine(self, x1=None, y1=None, x2=None , y2=None , p1=None , p2=None , line=None ): 
        self.prepareGeometryChange()
        
        if x1 and y1 and x2 and y2: 
            self._x1 = x1 
            self._y1 = y1 
            self._x2 = x2
            self._y2 = y2
            
        elif p1 and p2: 
            self._x1 = p1[0]
            self._y1 = p1[1] 
            self._x2 = p2[0] 
            self._y2 = p2[1] 
            
        elif line: 
            if len(line)==4: 
                self._x1 = line[0]
                self._y1 = line[1]
                self._x2 = line[2] 
                self._y2 = line[3]
                
scene = QGraphicsScene()
view = QGraphicsView() 

originMarker = QGraphicsEllipseItem(-10,-10,20,20) 
originMarker.setPen(QPen(Qt.magenta , 1))
scene.addItem(originMarker)

trace = TraceItem(30,50, 100,100)
# trace.setLine(100,100, 200,200)
trace.setPen(QPen(Qt.blue, 10 , c = Qt.PenCapStyle.RoundCap))
scene.addItem(trace)

view.setScene(scene) 
view.show() 
sys.exit(app.exec())

        

In [ ]:
l = [0,1,2,3]
print(l[0:2])
print(l[2:])

In [ ]:
# Reverifying Trace 

from PySide6.QtWidgets import *
from PySide6.QtCore import *
from PySide6.QtGui import *
from LayersItem import *
import sys
from MainWindow import MainWindow

from Trace import Trace

window = MainWindow()
boardScene = window.centralWidget().widget(1).scene()

line = QGraphicsLineItem(QLineF( 50,50 , 1000, 400))
trace = Trace( layers= ['F.Cu'] , traceWidth = 1 , p1 = QPointF(40,50 ) , p2 = QPointF(1000, 400) )
boardScene.addItem(trace)
boardScene.addItem(line)

window.show()
sys.exit(qApp.exec())

In [ ]:
# My Viewport dots are only appearing in one quadrant; xy positive. How do I get them in all quadrants? 
# A: offset by the exposed rect : painter.drawPoint( x*tickSpacing + rect.left()) , y*tickSpacing + rect.top()) 
# Great, but now there is a 'flowing' effect, appears as if dots flow behind items when zooming. (This because dots start drawing at rect leftTop. 
# A: Start drawing dots on grid, at grid pos nearest leftTop





In [ ]:
# My viewport is constrained to scroll only in a certain area. I want to be able to scroll on an effectively infinte area
# Answer: Set a large scroll area with :
#     view.setSceneRect(-10000, -10000, 20000, 20000)

# ### QGraphicsView ### 
# Visualize the contents of a QGraphicsScene in a scrollable viewport. 

# Scroll to any position on the scene using the scrollbars,(umm yet I cannot)(What is my sceen rectangle? unset, so infinite) or by calling .centerOn(QPoint)
# The visualized area is by default detected with QGS.itemsBoundingRect(), which returns the bounding rect of all items on the scene. 
# Use QGVIEW.setSceneRect() to set the visualized area yourself. This will adjust the scroll bars' ranges. 

# QGraphicsView.setBackground(painter, rect) default fills rect using the view's background brush. If no such brush defined(the default), the scene's .drawBackground is called instead. 

In [ ]:
# Set Background to Draw Dots Sensibly
from utils import * 
from MyView import MyView
from SchematicScene import SchematicScene

class GridView(MyView ): 


    def drawBackground(self, painter, rect): 

        print()
        print('DRAWBACKGROUND')

        painter.setBrush(Qt.black)
        painter.setPen(QPen(Qt.black, 1)) # Note QPen width 1 makes dots much more visible than width 0 
        # painter.setPen(Qt.NoPen) Makes points disappear
        


        # Note WHen zooming wayin to wayout , the PEN WIDTH of your painted points becomes important so the user can see it. 
        # I'm content with this for prototype, but production app should dynamically set pen widths based on zoom (?)
        def calculateTickSpacing():
            xScale = painter.transform().m11() # xScale is represented at the transformationMatrix m11 element. 
            print('XSCALE:', xScale)
            tickSpacing = Utils.boardTickSpacing
            if (xScale <.5): # ZOOMEDWAYOUT
                painter.setPen(QPen(Qt.black, 10)) # Set a wide pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing*10
            elif .5 <= xScale <= 10: 
                painter.setPen(QPen(Qt.black, 1))
                tickSpacing = Utils.boardTickSpacing
            if xScale > 20: #ZOOMEDWAYIN
                painter.setPen(QPen(Qt.black, .1)) # set a thin pen so user can still see the dots 
                tickSpacing = Utils.boardTickSpacing/10

            return tickSpacing

        tickSpacing = calculateTickSpacing()
        print('TICKSPACING:', tickSpacing)
        
        numTicksX = int(rect.width()/tickSpacing) 
        numTicksY = int(rect.height()/tickSpacing) 

        xStart = int(rect.left() / tickSpacing) * tickSpacing # important to start drawing points snapped to grid. If start drawing points at rect.left()&rect.top(), induces a 'flowing' effect while zooming
        yStart = int(rect.top() / tickSpacing) * tickSpacing
        
        for i in range(numTicksX): 
            for j in range(numTicksY):
                x = i*tickSpacing + xStart 
                y =  j* tickSpacing + yStart
                painter.drawPoint(QPointF(x , y)) # Note pass a QPointF() to be able to use floats with .drawPoint() 
                # painter.drawEllipse(QPointF(x,y), 1, 1)

    def wheelEvent(self, event): # Wheel as in mouseWheel 
        
        delta = event.angleDelta().y() # How much mouseWheel scrolled
        scaleFactor = math.pow(2.0, -delta / 500)
        self.scaleScene(scaleFactor)

    def scaleScene(self, scaleFactor):
        zoom = self.transform().scale(scaleFactor, scaleFactor).m11() # Scale current transform to predict zoom. The x scale lives in the matrix's m11 element      #  Used to do this , which also works: .mapRect(QRectF(0, 0, 1, 1)).width() # QTransform.mapRect(rect) -> QRectF, mapped onto the given QTransform. Note that we gave a unit rectangle; a rectangle where width&height=1. So, we are testing to see how much a unit scales under this transform. Note that self.transform() includes any previous scaling; representing the currently applied zoom, which we should limit to a certain range 

        if zoom < 0.01 or zoom > 100: # Prevent crazy scale changes.
            return

        self.scale(scaleFactor, scaleFactor) 



        
view = GridView() 
scene = SchematicScene()
view.setScene(scene)
r = QGraphicsRectItem(-100,-100, 200,200) 
r.setBrush(Qt.blue) 
r.setPen(Qt.NoPen)
scene.addItem(r)

view.show()

sys.exit(app.exec())






In [ ]:
# Workflow porting NetSym from Kicad: 3V3 symbol
from utils import * 
from kicadSymbolConverter import KicadSymbolConverter
# from kicadFootprintConverter import KicadFootprintConverter

officialKicadSymbolsLibrariesPath = os.path.join('third_party', 'kicad', 'symbols', 'kicad-symbols')

threeV3FilePath = os.path.join(officialKicadSymbolsLibrariesPath , 'power.kicad_symdir', '+3V3.kicad_sym')
threeV3SymFile = KicadSymbolConverter.convert(threeV3FilePath, categories = ['netSymbols']) # places '+3V3.sym' into the 'netSymbols' folder.Note 'categories so named bc supposed to be like ['capacitors', 'ceramic_capacitors']




